In [1]:
import cv2
import os

In [2]:
IMG_DIR = "car_dataset/images"
LBL_DIR = "car_dataset/labels"

CLASS_ID = 0
CLASS_NAME = "car"

os.makedirs(LBL_DIR, exist_ok=True)

print("Images folder:", IMG_DIR)
print("Labels folder:", LBL_DIR)

Images folder: car_dataset/images
Labels folder: car_dataset/labels


In [3]:
drawing = False
box_ready = False

ix, iy = -1, -1
end_x, end_y = -1, -1

original_img = None
display_img = None


def draw_rectangle(event, x, y, flags, params):

    global ix, iy, end_x, end_y
    global drawing, box_ready
    global display_img, original_img

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        box_ready = False
        ix, iy = x, y
        end_x, end_y = x, y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            end_x, end_y = x, y
            display_img = original_img.copy()

            cv2.rectangle(
                display_img,
                (ix, iy),
                (end_x, end_y),
                (0, 255, 0),
                2
            )

    elif event == cv2.EVENT_LBUTTONUP:

        drawing = False
        box_ready = True

        end_x, end_y = x, y

        display_img = original_img.copy()

        cv2.rectangle(
            display_img,
            (ix, iy),
            (end_x, end_y),
            (0, 255, 0),
            2
        )

print("Mouse callback ready")

Mouse callback ready


In [4]:
def convert_to_yolo(x1, y1, x2, y2, img_w, img_h):

    x_min = min(x1, x2)
    x_max = max(x1, x2)

    y_min = min(y1, y2)
    y_max = max(y1, y2)

    bw = x_max - x_min
    bh = y_max - y_min

    xc = x_min + bw / 2
    yc = y_min + bh / 2

    return (
        xc / img_w,
        yc / img_h,
        bw / img_w,
        bh / img_h
    )

print("YOLO conversion function ready")

YOLO conversion function ready


In [5]:
image_files = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
])

print("Total Images:", len(image_files))

Total Images: 89


In [ ]:
cv2.namedWindow("Car Annotation Tool")
cv2.setMouseCallback(
    "Car Annotation Tool",
    draw_rectangle
)

idx = 0

while idx < len(image_files):

    image_name = image_files[idx]

    image_path = os.path.join(
        IMG_DIR,
        image_name
    )

    original_img = cv2.imread(image_path)
    original_img = cv2.resize(
    original_img,
    (1200, 800)
    )

    if original_img is None:
        idx += 1
        continue

    img_h, img_w = original_img.shape[:2]

    display_img = original_img.copy()

    box_ready = False

    while True:

        preview = display_img.copy()

        cv2.putText(
            preview,
            f"[{idx+1}/{len(image_files)}] {image_name}",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        cv2.imshow(
            "Car Annotation Tool",
            preview
        )

        key = cv2.waitKey(20) & 0xFF

        if key == ord("s"):

            if box_ready:

                xc, yc, bw, bh = convert_to_yolo(
                    ix, iy,
                    end_x, end_y,
                    img_w, img_h
                )

                label_name = (
                    os.path.splitext(
                        image_name
                    )[0]
                    + ".txt"
                )

                label_path = os.path.join(
                    LBL_DIR,
                    label_name
                )

                with open(label_path, "w") as f:

                    f.write(
                        f"{CLASS_ID} "
                        f"{xc:.6f} "
                        f"{yc:.6f} "
                        f"{bw:.6f} "
                        f"{bh:.6f}\n"
                    )

                print(
                    "Saved:",
                    label_path
                )

                idx += 1
                break

        elif key == ord("n"):
            idx += 1
            break

        elif key == ord("q"):
            idx = len(image_files)
            break

cv2.destroyAllWindows()

print("Annotation Complete")

In [8]:
label_files = [
    f for f in os.listdir(LBL_DIR)
    if f.endswith(".txt")
]

print(
    "Total Labels Saved:",
    len(label_files)
)

if label_files:

    sample = label_files[0]

    print("\nSample Label File:")

    with open(
        os.path.join(
            LBL_DIR,
            sample
        )
    ) as f:

        print(f.read())

Total Labels Saved: 89

Sample Label File:
0 0.399583 0.650000 0.599167 0.295000

